# Assignment 3: Airbnb Investment Dashboard 
### David Hook - SIADS 521
---
This notebook demonstrated the techniques I used to build an interactive Plotly Dash dashboard designed to help my wife explore the best US cities for short-term rental income through Airbnb. The dashboard seeks to explore the relationship between the median price of houses in the market and the volume and revenue of rentals in those regions. The goal was to address the full scope of assignment 3, gain some practice in regional heatmaps, and 

## Section 1 — Visualization Technique

### 1.1 Chart Types Used in the Dashboard

**Horizontal Bar Chart** — Ranks the top cities on a single metric (gross yield, occupancy, or revenue). A dropdown  in the graph lets the user switch metrics, so the same chart answers several questions.

**Scatter Plot** — Each dot is a city positioned by home price (x) vs. annual Airbnb revenue (y), with dot size encoding listing count. A 5 % yield reference line gives investors a quick pass/fail benchmark for that real estate market.

**Box Plot** — Shows the distribution of monthly revenue within each city. The box captures the middle 50 % of listings; outlier dots flag unusually high or low earners. Narrow boxes mean predictable income; wide boxes mean higher risk.

**Choropleth Map with Bubble Overlay** — A filled map colored by a continuous variable (state-level home prices) with proportionally-sized circles encoding a second variable (listing density per city). This displays geographic patterns at two levels of detail simultaneously.

### 1.2 How the Visualizations Complement Each Other

| Chart | Question it answers |
|-------|-------------------|
| Choropleth + Bubbles | Where is demand, and how expensive is each market? |
| Scatter Plot | Which cities give the best return relative to home cost? |
| Bar Chart | Which city ranks #1 on a chosen metric? |
| Box Plot | How variable/risky is the revenue in each city? |

### 1.3 Dashboard Interactivity

All of the dashboards have interactive hover elements with labels for variables of interest. Additionally, the drop down menu selects a metric of interest and ranks all 27 markets by that metric, and those cities are refreshed in the scatter and box plots so we can see how they perform on those measures. 

---
## Section 2 — Visualization Library

### 2.1 Framework and Libraries

This project uses four main packages:

* **Plotly**:  Core charting functions to create interactive charts 
* **PlotlyExpress**: High-level API for common chart types
* **Dash**: Python web framework that turns Plotly figures into a web dashboard 
* **ipywidgets**: Creates interactive widgets for Jupyter notebooks 

Plotly and Plotly Express were created by **Plotly Inc.** (Montreal, founded 2013) and is fully **open-source** under the MIT license. Dash, also from Plotly Inc., is also MIT-licensed. ipywidgets is a Jupyter project also released under an open-source license.

All libraries are installed with pip:

```bash
pip install plotly 
pip install dash 
pip install ipywidgets
```

**Why Plotly + Dash + ipywidgets is suitable for this project:** Plotly is one of the easiest to use libraries for building interactive web visualizations in Python. Dash extends plotly's capabilities letting us place multiple Plotly charts on a single page with shared controls. ipywidgets bridges the gap between a static notebook and a full Dash app — we can wire a single dropdown to update two separate Plotly figures in real time, which Plotly's built-in `updatemenus` cannot do on its own. The combination covers the full requirements for this week's assignment.

### 2.2 General Approach and Limitations

**Declarative and procedural.** Plotly Express is declarative, however when finer control is needed (i.e. layering the`go.Choropleth` and `go.Scattergeo` on the same figure) we can take advantage of the procedural `graph_objects` API. Having both abilities in one library lets us keep simple charts stay simple, while complex charts are still possible.

**Limitations:**

- **Performance on large data** — Plotly sends the full dataset to the browser as JSON. Our ~136K rows work fine, but larger datasets might benefit from a different approach.
- **Requires a running server** — a Dash dashboard needs a Python process (locally or on a host like Render), which adds a deployment step compared to a static HTML export.
- **ipywidgets are notebook-only** — `widgets.Dropdown` and `widgets.Output` work inside Jupyter but won't render in a standalone HTML export or a Dash app. For the deployed dashboard, the same interactivity is handled by Dash callbacks instead.

---
# Section 3 — Demonstration

## 3.0 Data Sources

This dashboard combines two publicly available datasets:

**Airbnb listings** — The file `AB_US_2023.csv` comes from Kritik Seth's [U.S. Airbnb Open Data](https://www.kaggle.com/datasets/kritikseth/us-airbnb-open-data) dataset on Kaggle. It is a compilation of scraped listings from [Inside Airbnb](http://insideairbnb.com/) covering 27 US cities, last updated April 2023. The Kaggle page was originally created for unrelated exploration tasks (predicting listing prices, recommending titles to hosts, describing regions by listing names), but the underlying per-listing data is perfectly suited for the investment dashboard as well.

**Home prices** — Zillow's [ZHVI (Zillow Home Value Index)](https://www.zillow.com/research/data/) provides monthly median home values at the city and metro level. We downloaded the latest city-level and metro-level CSVs directly from Zillow Research. These give us a reliable home-cost baseline to pair against Airbnb revenue estimates.

**Limitations to keep in mind:** The Airbnb data is from 2023 and covers only 27 cities, so the dashboard should not be treated as current investment advice. Howver, the dataset is large enough  and spans enough geography to demonstrate the visualization techniques and interactive dashboard features that are the focus of this assignment.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

---
## 3.1 Loading and Cleaning the Airbnb Data

The raw CSV has ~232K rows (one per listing) across 27 US cities. We only need whole-home rentals with reasonable prices, so we filter to `room_type == "Entire home/apt"` and keep prices between $10 and $2,000/night to remove data-entry errors and extreme luxury outliers. We also trim to just the columns the dashboard actually uses.

In [2]:
# Load the raw Kaggle CSV
raw = pd.read_csv("data/raw/AB_US_2023.csv", low_memory=False)
print(f"Raw dataset: {len(raw):,} rows, {raw['city'].nunique()} cities")

# Filter to whole-home rentals and remove price outliers
df = raw[raw["room_type"] == "Entire home/apt"].copy()
df = df[(df["price"] >= 10) & (df["price"] <= 2000)]

# Keep only the columns the dashboard uses
df = df[["city", "price", "room_type", "neighbourhood",
         "latitude", "longitude", "reviews_per_month",
         "availability_365", "minimum_nights"]].copy()

Raw dataset: 232,147 rows, 27 cities


---
## 3.2 Adding Zillow Home Prices

To calculate investment yield we need each city's median home price. Zillow publishes monthly ZHVI data at both city and metro level. The main challenge is that the two datasets use different city names — for example, Airbnb says "New York City" while Zillow says "New York", and Airbnb uses county names like "Broward County" where Zillow lists "Fort Lauderdale". We handle this with a manual mapping dictionary, then look up the most recent ZHVI value for each of our 27 cities.

In [3]:
# Load Both Zillow ZHVI CSVs (city-level and metro-level)
zillow_city = pd.read_csv("data/raw/zillow_zhvi_city.csv")
zillow_metro = pd.read_csv("data/raw/zillow_zhvi_metro.csv")
latest_city_col = zillow_city.columns[-1]
latest_metro_col = zillow_metro.columns[-1]

# Map Airbnb city names → Zillow lookup keys
# Format: "Airbnb name": ("Zillow RegionName", "State", "city"|"metro")
airbnb_to_zillow = {
    "Asheville":          ("Asheville", "NC", "city"),
    "Austin":             ("Austin", "TX", "city"),
    "Boston":             ("Boston", "MA", "city"),
    "Broward County":     ("Fort Lauderdale", "FL", "city"),
    "Cambridge":          ("Cambridge", "MA", "city"),
    "Chicago":            ("Chicago", "IL", "city"),
    "Clark County":       ("Las Vegas", "NV", "city"),
    "Columbus":           ("Columbus", "OH", "city"),
    "Denver":             ("Denver", "CO", "city"),
    "Jersey City":        ("Jersey City", "NJ", "city"),
    "Los Angeles":        ("Los Angeles", "CA", "city"),
    "Nashville":          ("Nashville", "TN", "city"),
    "New Orleans":        ("New Orleans", "LA", "city"),
    "New York City":      ("New York", "NY", "city"),
    "Oakland":            ("Oakland", "CA", "city"),
    "Pacific Grove":      ("Pacific Grove", "CA", "city"),
    "Portland":           ("Portland", "OR", "city"),
    "Rhode Island":       ("Providence, RI", None, "metro"),
    "Salem":              ("Salem", "OR", "city"),
    "San Diego":          ("San Diego", "CA", "city"),
    "San Francisco":      ("San Francisco", "CA", "city"),
    "San Mateo County":   ("San Mateo", "CA", "city"),
    "Santa Clara County": ("Santa Clara", "CA", "city"),
    "Santa Cruz County":  ("Santa Cruz", "CA", "city"),
    "Seattle":            ("Seattle", "WA", "city"),
    "Twin Cities MSA":    ("Minneapolis, MN", None, "metro"),
    "Washington D.C.":    ("Washington", "DC", "city"),
}

# Look up the most recent ZHVI value for each Airbnb city
zillow_prices = {}
for airbnb_name, (zillow_name, state, source) in airbnb_to_zillow.items():
    if source == "city":
        match = zillow_city[
            (zillow_city["RegionName"] == zillow_name) & (zillow_city["State"] == state)
        ]
        if len(match) == 1:
            zillow_prices[airbnb_name] = round(match[latest_city_col].values[0])
    else:
        match = zillow_metro[zillow_metro["RegionName"] == zillow_name]
        if len(match) == 1:
            zillow_prices[airbnb_name] = round(match[latest_metro_col].values[0])

# Merge onto Airbnb data and drop cities without a Zillow match
df["zillow_home_price"] = df["city"].map(zillow_prices)
df = df.dropna(subset=["zillow_home_price"]).copy()

---
## 3.3 Derived Columns and City Summary

We derive occupancy rate from `availability_365`, estimate revenue at multiple timeframes, and calculate gross yield (annual revenue / home price). Then we aggregate to a city-level summary and compute a composite investment score that ranks cities by yield, occupancy, and market saturation.

In [4]:
# Derived columns
df["occupancy_rate"] = ((365 - df["availability_365"]) / 365).round(3)
df["nightly_revenue"] = df["price"]
df["weekly_revenue"] = df["price"] * 7
df["monthly_revenue"] = (df["price"] * 30 * df["occupancy_rate"]).round(0)
df["annual_revenue"] = (df["price"] * 365 * df["occupancy_rate"]).round(0)
df["gross_yield"] = (df["annual_revenue"] / df["zillow_home_price"]).round(4)

# City-level summary (all 27 cities)
city_stats = df.groupby("city").agg(
    median_price=("price", "median"),
    occupancy_rate=("occupancy_rate", "median"),
    median_annual_revenue=("annual_revenue", "median"),
    total_reviews=("reviews_per_month", "sum"),
    listing_count=("price", "count"),
    zillow_home_price=("zillow_home_price", "first"),
).reset_index()
city_stats["gross_yield"] = (
    city_stats["median_annual_revenue"] / city_stats["zillow_home_price"]
).round(4)

# Investment score: average of yield rank, occupancy rank, and inverse listing-count rank
n = len(city_stats)
city_stats["investment_score"] = (
    (city_stats["gross_yield"].rank() / n +
     city_stats["occupancy_rate"].rank() / n +
     (1 - city_stats["listing_count"].rank() / n)) / 3
).round(3)

print(f"{len(df):,} listings across {n} cities")
print(f"Top 10 selection happens interactively in Chart 1 below")

167,855 listings across 27 cities
Top 10 selection happens interactively in Chart 1 below


---
# Section 4 — Building Each Visualization

Now comes the fun part. We'll build each of the dashboard charts one at a time using **Plotly Express** (`px`). 

Plotly Express is the simplest way to make Plotly charts. The pattern is always the same:

```python
fig = px.chart_type(
    data,               # your DataFrame
    x="column_name",    # what goes on the x-axis
    y="column_name",    # what goes on the y-axis
    title="Chart Title" # the title shown above the chart
)
fig.show()              # display it
```

Every chart is interactive — you can hover, zoom, and pan automatically.

### 4.1 - Chart 1: Interactive Bar Chart — City Rankings

This chart uses an ipywidgets dropdown to let the user pick a metric. Changing the selection re-ranks the top 10 cities and redraws the bar chart. It also sets the global `top10` and `top_df` variables so that Charts 2 and 3 below reflect the same city selection when run. In the dashboard they will update automatically.

* Define the metrics you want to rank by as a dictionary.
  - `key`: display name for the dropdown (e.g. `"Gross Yield"`)
  - `value`: a dict with `col` (column to sort by) and `label` (axis label for the chart)
* Build the dropdown using `widgets.Dropdown()`.
  - `options`: list of metric names pulled from the dict keys
  - `value`: which option is selected by default (`"Gross Yield"`)
  - `description`: text label shown next to the dropdown
  - `style`: `{"description_width": "auto"}` so the label doesn't get clipped
* Create a `widgets.Output()` to hold the chart area.
* Define a `render_bar()` that:
  - Grabs the selected metric's `col` and `label` from the dict
  - Picks the top 10 cities for that metric
  - Filters `df` down to only listings in `top_df`
  - Assigns each city a color 
  - Builds a horizontal bar chart with `px.bar(..., orientation="h")`
* Connect the dropdown using `.observe(render_bar, names="value")`
* Display both widgets and call `render_bar()` to draw the initial chart

In [ ]:
# --- Chart 1: Interactive Horizontal Bar Chart ---

metrics = {
    "Gross Yield":          {"col": "gross_yield",           "label": "Gross Yield (annual revenue / home price)"},
    "Occupancy Rate":       {"col": "occupancy_rate",        "label": "Occupancy Rate"},
    "Annual Revenue ($)":   {"col": "median_annual_revenue", "label": "Median Annual Revenue ($)"},
    "Median Nightly Price": {"col": "median_price",          "label": "Median Nightly Price ($)"},
    "Home Price ($)":       {"col": "zillow_home_price",     "label": "Zillow Median Home Price ($)"},
    "Listing Count":        {"col": "listing_count",         "label": "Number of Listings"},
    "Investment Score":     {"col": "investment_score",       "label": "Investment Score"},
}

metric_dropdown = widgets.Dropdown(
    options=list(metrics.keys()),
    value="Gross Yield",
    description="Rank by:",
    style={"description_width": "auto"},
)

bar_output = widgets.Output()

def render_bar(_=None):
    global top10, top_df, city_colors

    name = metric_dropdown.value
    col = metrics[name]["col"]
    label = metrics[name]["label"]

    # Select and sort top 10 cities by the chosen metric
    top10 = city_stats.nlargest(10, col).reset_index(drop=True)
    top_df = df[df["city"].isin(top10["city"])]
    sorted_data = top10.sort_values(col, ascending=True)

    # Assign a consistent color to each city (shared with Charts 2 & 3)
    palette = px.colors.qualitative.Plotly
    city_colors = {city: palette[i % len(palette)]
                   for i, city in enumerate(top10.sort_values(col, ascending=False)["city"])}

    with bar_output:
        clear_output(wait=True)
        fig_bar = px.bar(
            sorted_data,
            x=col, y="city", orientation="h",
            title=f"Top 10 Cities by {name}",
            labels={col: label, "city": ""},
            color="city", color_discrete_map=city_colors,
        )
        fig_bar.update_traces(hovertemplate="%{y}: %{x}<extra></extra>")
        fig_bar.update_yaxes(categoryorder="array",
                             categoryarray=sorted_data["city"].tolist())
        fig_bar.update_layout(height=400, showlegend=False, title_x=0.5)
        fig_bar.show()

metric_dropdown.observe(render_bar, names="value")
display(metric_dropdown, bar_output)
render_bar()

Dropdown(description='Rank by:', options=('Gross Yield', 'Occupancy Rate', 'Annual Revenue ($)', 'Median Night…

Output()

### 4.2 - Chart 2: Scatter Plot — Home Price vs Revenue

Each dot is one of the top 10 cities from Chart 1. The plot shows whether cities with expensive homes also generate more Airbnb revenue. A 5% yield line gives a baseline for acceptable investment return. A 5-8% annual yield is considered a safe target range, I selected the bottom of the acceptable range as the line to beat. 

* Build a scatter plot with `px.scatter()` on the `top10` DataFrame
  - `x`: `zillow_home_price` — median home price on the horizontal axis
  - `y`: `median_annual_revenue` — median annual Airbnb revenue on the vertical axis
  - `size`: `listing_coun` — bubble area scales with number of listings
  - `text`: `city` — label each dot with the city name
  - `title`: chart title with subtitle noting what bubble size means
  - `labels`: dict that remaps column names to readable axis/legend labels
    - `zillow_home_price' to Median Home Price
    - `median_annual_revenue`to `Median Annual Revenue ($)`
    - `listing_count` to `"isting Count` 
* Add a 5% yield reference line with `go.Scatter()`
  - `x`: `[0, max_price]` — line spans the full price range
  - `y`: `[0, max_price * 0.05]` — 5% of each x value
  - `mode`: `"lines"` — no markers
  - `name`: `"5% Yield Line"` — label in the legend
  - `line`: `dict(dash="dash", color="gray")` — dashed gray style
* Position city labels with `update_traces()`
  - `textposition`: `"top center"` — place labels above each dot
  - `selector`: `dict(mode="markers+text")` — only apply to the scatter trace, not the line
* Final layout with `update_layout()`

In [6]:
# --- Chart 2: Scatter Plot ---

fig_scatter = px.scatter(
    top10,
    x="zillow_home_price",
    y="median_annual_revenue",
    size="listing_count",
    text="city",
    title="Home Price vs Annual Revenue (Top 10 Cities)<br><sup>Bubble size = number of Airbnb listings</sup>",
    labels={
        "zillow_home_price": "Median Home Price ($)",
        "median_annual_revenue": "Median Annual Revenue ($)",
        "listing_count": "Listing Count",
    },
)

# Add a 5% yield reference line
max_price = top10["zillow_home_price"].max() * 1.1

fig_scatter.add_trace(go.Scatter(
    x=[0, max_price],
    y=[0, max_price * 0.05],
    mode="lines",
    name="5% Yield Line",
    line=dict(dash="dash", color="gray"),
))

fig_scatter.update_traces(
    textposition="top center",
    selector=dict(mode="markers+text")
)

fig_scatter.update_layout(height=500, showlegend=True, title_x=0.5)
fig_scatter.show()

### 4.3 -  Chart 3: Box Plot — Revenue Distribution

Shows the spread of monthly revenue for each of the top 10 cities from Chart 1. A city with a huge box is riskier. A narrow box means most listings earn about the same.

* Get the current ranking order so the x-axis matches Chart 1
  - `current_col`: grabs the column name for the metric currently selected in the dropdown
  - `city_order`: sorts `top10` descending by that column and pulls city names into a list
* Build a box plot with `px.box()` on `top_df` (all individual listings in the top 10 cities)
  - `x`: `city` — one box per city
  - `y`: `monthly_revenue` — the value to show the distribution of
  - `title`: title includes current metric name
  - `labels`: dict remapping column names to readable text
    - `monthly_revenue` to `Monthly Revenue`
    - `city` to "" (hides the extra axis label)
  - `color`: `city` — color each box differently
  - `color_discrete_map`: `city_colors` — reuses the same palette assigned in Chart 1
  - `category_orders`: `{"city": city_order}` — forces the x-axis to match the bar chart ranking
* Cap the y-axis at the 99th percentile with `update_yaxes()`
  - `range`: `[0, y_cap * 1.05]` — cuts off extreme outliers so the boxes stay readable
* Final layout with `update_layout()`

In [9]:
# Rank order matches the current bar chart
current_col = metrics[metric_dropdown.value]["col"]
city_order = top10.sort_values(current_col, ascending=False)["city"].tolist()

print(f"Showing cities ranked by: {metric_dropdown.value}")
print(f"Listings in top 10 cities: {len(top_df):,}")

fig_box = px.box(
    top_df,
    x="city", y="monthly_revenue",
    title=f"Monthly Revenue Distribution — Top 10 by {metric_dropdown.value}",
    labels={"monthly_revenue": "Monthly Revenue ($)", "city": ""},
    color="city",
    color_discrete_map=city_colors,
    category_orders={"city": city_order},
)

# Cap y-axis at the 99th percentile 
y_cap = top_df["monthly_revenue"].quantile(0.99)
fig_box.update_yaxes(range=[0, y_cap * 1.05])
fig_box.update_layout(height=450, showlegend=False, xaxis_tickangle=-45, title_x=0.5)
fig_box.show()

Showing cities ranked by: Investment Score
Listings in top 10 cities: 40,536


### Chart 4: Choropleth + Bubbles — Geographic Overview

Layers two `plotly.graph_objects` traces on one figure to answer: "Where is real estate expensive?" and "Where is Airbnb demand high?" 

* Aggregate state-level median home prices from the Zillow dataset.
  - Group `zillow_city` by `State`, take the median of the latest price column
  - Rename the result to `median_home_price`
* Aggregate city-level bubble data from our Airbnb listings with `df.groupby("city").agg()`.
  - `lat`: mean of `latitude` — center point for the bubble
  - `lon`: mean of `longitude` — center point for the bubble
  - `listing_count`: count of `price` — number of listings (drives bubble size)
  - `zillow_home_price`: first value — home price for that city
* Create the figure with `go.Figure()`.
* Add Layer 1 with `go.Choropleth()` — colors each state by median home price.
  - `locations`: state abbreviation column 
  - `z`: the numeric price values that select the color
  - `locationmode`: `USA-states` — match on state abbreviations
  - `colorscale`: `YlOrRd` — yellow-to-red gradient
  - `colorbar`: dict with `title`, `x` position, and dollar formatting
  - `hovertemplate`: showing state name and formatted price
* Add Layer 2 with `go.Scattergeo()` — one bubble per Airbnb city.
  - `lat`, `lon`: from the aggregated city data
  - `text`: city name
  - `marker`: dict controlling bubble appearance
    - `size`: `listing_count` values — raw counts that Plotly scales into pixel sizes
    - `sizemode`: `area` — scales bubble based on area 
    - `sizeref`: set the size for the largest bubble
    - `sizemin`: `4` — smallest bubble won't shrink below 4px
    - `color`: rgb+alpha color
    - `line`: `dict(width=1, color="white")` 
  - `hovertemplate`: shows city name and listing count on hover
  - `showlegend`: `False` — bubbles don't need a legend entry because we explain their encoding in subtitle
* Final layout with `update_layout()`.

In [8]:
# --- Chart 4: Home Prices Choropleth + Listing Density Bubbles ---

# 1) Build state-level median home prices from the full Zillow dataset
state_prices = (
    zillow_city.groupby("State")[latest_city_col]
    .median()
    .reset_index()
    .rename(columns={latest_city_col: "median_home_price"})
)

# 2) Build city-level bubble data from our Airbnb listings
city_bubbles = df.groupby("city").agg(
    lat=("latitude", "mean"),
    lon=("longitude", "mean"),
    listing_count=("price", "count"),
    zillow_home_price=("zillow_home_price", "first"),
).reset_index()

# 3) Combine both layers into one figure
fig_combo = go.Figure()

# Layer 1: Choropleth — state-level real estate prices
fig_combo.add_trace(go.Choropleth(
    locations=state_prices["State"],
    z=state_prices["median_home_price"],
    locationmode="USA-states",
    colorscale="YlOrRd",
    colorbar=dict(
        title="Median Home<br>Price ($)",
        x=1.0,
        tickformat="$,.0f",
    ),
    hovertemplate="<b>%{location}</b><br>Median Home Price: $%{z:,.0f}<extra></extra>",
))

# Layer 2: Bubbles — one per Airbnb city, sized by listing count
fig_combo.add_trace(go.Scattergeo(
    lat=city_bubbles["lat"],
    lon=city_bubbles["lon"],
    text=city_bubbles["city"],
    marker=dict(
        size=city_bubbles["listing_count"],
        sizemode="area",
        sizeref=2.0 * city_bubbles["listing_count"].max() / (40**2),
        sizemin=4,
        color="rgba(0, 100, 200, 0.6)",
        line=dict(width=1, color="white"),
    ),
    hovertemplate=(
        "<b>%{text}</b><br>"
        "Listings: %{marker.size:,}<br>"
        "<extra></extra>"
    ),
    showlegend=False,
))

fig_combo.update_layout(
    title="US Real Estate Prices with Airbnb Listing Density<br><sup>Bubble size = number of Airbnb listings</sup>",
    title_x=0.5,
    geo=dict(
        scope="usa",
        showlakes=True,
        lakecolor="rgb(200, 220, 240)",
    ),
    height=550,
    margin=dict(l=0, r=0, t=60, b=0),
)

fig_combo.show()

---
## 4.2 How the Charts Work Together in the Dashboard

| Chart | Question it answers | Data used |
|-------|-------------------|-----------|
| Bar Chart (interactive) | Which cities rank highest on a chosen metric? | Top 10 from `city_stats` (selection drives Charts 2 & 3) |
| Scatter Plot | Which cities give the best return vs home cost? | Top 10 city summary |
| Box Plot | How variable/risky is the revenue in each city? | Individual listings for the same top 10 cities |
| Choropleth + Bubbles | Where is demand, and how expensive is each market? | State-level Zillow prices + city-level listings |

In the full Dash dashboard, all charts share the same **price range filter**. When you move the slider, every chart updates to show only listings in that price range. This is handled by Dash **callbacks** — Python functions that run automatically when a user interacts with a widget. The `ipywidgets` dropdown in Chart 1 demonstrates the same concept inside a notebook — change the metric, then re-run Charts 2 and 3 to see the updated city selection.

---
# Section 5 — Dashboard Deployment

The deployed dashboard can be viewed at: <URL>

The code repository can be found at: <URL>


# Section 6 - Reflection

I would have liked to explored transformations a bit more with this dashboard. The lecture in Module 3 specifically mentioned real estate as an opportunity to do that and it might have changed the rankings in a meaningful way. Unfortunately I ran out of time exploring widgets and learning how to do the Choropleth, but that was the main chart I wanted to gain experience with on this assignment.  